# LangGraph Stateful Multi-Agent Workflows

Demonstrates stateful agent graph workflows using langgraph.graph.StateGraph and ChatGoogleGenerativeAI (gemini-3.6-flash).

In [1]:
import os
from typing import TypedDict, Annotated, Sequence
from dotenv import load_dotenv, find_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph import StateGraph, START, END

load_dotenv(find_dotenv())
llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash")

class AgentState(TypedDict):
    messages: Sequence[BaseMessage]

def chatbot_node(state: AgentState):
    response = llm.invoke(state["messages"])
    return {"messages": list(state["messages"]) + [response]}

builder = StateGraph(AgentState)
builder.add_node("chatbot", chatbot_node)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

graph = builder.compile()
result = graph.invoke({"messages": [HumanMessage(content="Hi, introduce yourself as a LangGraph assistant.")]})

print(f"LangGraph Execution Output:\n{result['messages'][-1].content}")


ChatGoogleGenerativeAIError: Error calling model 'gemini-3.6-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash\nPlease retry in 56.985339777s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT